In [1]:
import pandas as pd
import numpy as np
import xarray as xr
import h5py
from datetime import datetime
from metpy.calc import wind_components,wind_direction,wind_speed
from metpy.units import units

In [2]:
def format_datetime64(val, format_str='%Y-%m-%d %H:%M:%S'):
    dt = pd.to_datetime(str(val))
    return dt.strftime(format_str)

## Script settings

In [3]:
scope = 'H3'

project_root = '/work/bb1461/'
exp = 'v370_2030'

In [4]:
if scope == 'H2':
    scope_name = 'HEFEX II'
    obs_file = f'hefex2_aws.h5'
    dt_from,dt_to = pd.to_datetime('2023-08-15', utc=True),pd.to_datetime('2023-09-10', utc=True)
elif scope == 'H3':
    scope_name = 'HEFEX III'
    obs_file = f'hefex3_aws.h5'
    dt_from,dt_to = pd.to_datetime('2025-08-05', utc=True),pd.to_datetime('2025-09-05', utc=True)
else:
    print('Invalid scope!')

loc_file = f'{scope}_Coords_w_cells.csv'

## Load and prepare location dimension

In [5]:
df_locs = pd.read_csv(loc_file).sort_values('Alt')
df_locs = df_locs[df_locs["Type"].str.startswith("AWS")]
df_locs

,cell_id,topo,Original Name,New Name,Type,UAV Takeoff,Lat,Lon,Alt,Comment,Alt_DEM_2022
1,34121,2426.978271,S2,A244,AWS,NaN,46.821410,10.805850,2438.0,from field log,2443.03
3,35219,2523.569824,S1,A245,AWS,NaN,46.816548,10.794520,2451.0,from field log,2456.78
8,35704,2798.971191,Tower,T272,AWS,NaN,46.801811,10.771261,2722.0,from SmartFlux GPS,2725.08
0,30329,3052.749023,HEF30,T303,AWS,NaN,46.791000,10.748700,3030.0,from UIBK Website,3029.45
13,36210,3038.568359,Hintereis,STHE,AWS,NaN,46.798896,10.760373,3031.0,from UIBK Website,3032.07
15,36558,3261.780029,Im hinteren Eis,IHE,AWS,NaN,46.795761,10.783409,3264.0,from UIBK Website,NaN


In [6]:
dim_loc = df_locs['New Name']
dim_loc = dim_loc.replace('Hintereis','STHE')

dim_loc_lat  = np.array(df_locs['Lat'])
dim_loc_lon  = df_locs['Lon']
dim_loc_type = df_locs['Type']
dim_loc_ele  = df_locs['Alt']

## Extract time series of each station for each variable

(variables have different names at each location)

In [7]:
# define column mapping for wind information
if scope == 'H2':
    wind_col_suffix = {
        'Hintereis': '_30',
        'A255'     : '_30_momaa',
        'A260'     : '_30_momaa',
        'A267R'    : '_30_momaa',
        'A267L'    : '_30_momaa',
        'A267'     : '_30_momaa',
        'D269'     : '_25',
        'T275'     : '_20_ec',
        'D281'     : '_25',
        'D293'     : '_25',
        'T303'     : '_20_ec',
        'A309'     : '_20_ec',
        'D316'     : '_25',
    }
elif scope == 'H3':
    wind_col_suffix = {
        'STHE' : '_30',
        'T303' : '_55',
    }
else:
    wind_col_suffix = {}


In [8]:
keys = h5py.File(obs_file, 'r').keys()
for k in keys:
    print(k,pd.DataFrame(pd.read_hdf(obs_file, key=k)).columns)

A244 Index(['T_20', 'RH_20', 'H2O', 'cell_press'], dtype='object')
A245 Index(['T_20', 'RH_20', 'H2O', 'cell_press'], dtype='object')
IHE Index(['T_15', 'RH_15', 'P'], dtype='object')
STHE Index(['T_20', 'RH_20', 'P', 'wdir_30', 'wspd_30'], dtype='object')
T272 Index(['T_20', 'RH_20', 'P_10', 'QV_10'], dtype='object')
T303 Index(['T_25', 'RH_25', 'P', 'wdir_55', 'wspd_55'], dtype='object')


In [9]:
locs = np.array(df_locs['New Name'])

df_temp = pd.DataFrame()
df_pres = pd.DataFrame()
df_rh = pd.DataFrame()
df_wspd = pd.DataFrame()
df_wdir = pd.DataFrame()
df_u = pd.DataFrame()
df_v = pd.DataFrame()
df_wspd_avg = pd.DataFrame()
df_wdir_avg = pd.DataFrame()
df_u_avg = pd.DataFrame()
df_v_avg = pd.DataFrame()

keys = h5py.File(obs_file, 'r').keys()
for loc in locs:
    if loc not in keys:
        print(f'Location {loc} not in .h5 file')
        continue
    
    print('+', loc, end='\t')
    df = pd.DataFrame(pd.read_hdf(obs_file, key=loc)) # make sure it becomes a DataFrame, not Series
    df_hr = df[df.index.minute == 0]
    
    # construct search terms for wind columns
    c_wspd, c_wdir = None, None
    if loc in wind_col_suffix.keys():       
        c_wspd = 'wspd' + wind_col_suffix[loc]
        c_wdir = 'wdir' + wind_col_suffix[loc]
        if c_wspd.endswith('momaa'):
            c_wspd = c_wspd + '_vector'        

    # determine column names for Temp, RH, Pres, Wind
    col_t, col_p, col_rh, col_wspd, col_wdir = None,None,None,None,None
    for c in df.columns:
        if c.startswith('T_20_log'):
            col_t = c
        elif (c.startswith('T_2') or c.startswith('T_15')) and col_t is None:
            # fallback for temp
            col_t = c
        elif c.endswith('_press'):
            df[c] = df[c] * 10  # cell pressure from irgason (HF)            
            col_p = c
        elif c.startswith('P') or c.startswith('p'):
            col_p = c
        elif c.startswith('RH_2') or c.startswith('RH_15'):
            col_rh = c
        elif c_wspd is not None and (c == c_wspd or c == 'FF'):
            col_wspd = c
        elif c_wdir is not None and c == c_wdir:
            col_wdir = c

    print('TEMP:',col_t, end='\t')
    print('PRES:',col_p, end='\t')
    print('RH:'  ,col_rh, end='\t')
    print('WSPD:',col_wspd, end='\t')
    print('WDIR:',col_wdir)

    if col_t is not None:
        df_temp = df_temp.join(df_hr[col_t].dropna().rename(loc), how="outer")
    if col_p is not None:
        df_pres = df_pres.join(df_hr[col_p].dropna().rename(loc), how="outer")
    if col_rh is not None:
        df_rh = df_rh.join(df_hr[col_rh].dropna().rename(loc), how="outer")
    if col_wspd is not None:
        df_wspd = df_wspd.join(df_hr[col_wspd].dropna().rename(loc), how="outer")
    if col_wdir is not None:
        df_wdir = df_wdir.join(df_hr[col_wdir].dropna().rename(loc), how="outer")

    # calc wind components
    if col_wspd is not None and col_wdir is not None:
        df = df[[col_wspd,col_wdir]]
        df['u'], df['v'] = wind_components(df[col_wspd].values * units('m/s'),
                                           df[col_wdir].values * units.deg)
        df_hr = df[df.index.minute == 0]
        df_u = df_u.join(df_hr['u'].dropna().rename(loc), how="outer")
        df_v = df_v.join(df_hr['v'].dropna().rename(loc), how="outer")
        
        df_30min = df.resample('30min', origin='end_day').mean()
        df_30min['wspd'] = wind_speed(df_30min['u'].values * units('m/s'), df_30min['v'].values * units('m/s'))
        df_30min['wdir'] = wind_direction(df_30min['u'].values * units('m/s'), df_30min['v'].values * units('m/s'))
                
        df_u_avg = df_u_avg.join(df_30min['u'].dropna().rename(loc), how="outer")
        df_v_avg = df_v_avg.join(df_30min['v'].dropna().rename(loc), how="outer")
        df_wspd_avg = df_wspd_avg.join(df_30min['wspd'].dropna().rename(loc), how="outer")
        df_wdir_avg = df_wdir_avg.join(df_30min['wdir'].dropna().rename(loc), how="outer")

+ A244	TEMP: T_20	PRES: cell_press	RH: RH_20	WSPD: None	WDIR: None
+ A245	TEMP: T_20	PRES: cell_press	RH: RH_20	WSPD: None	WDIR: None
+ T272	TEMP: T_20	PRES: P_10	RH: RH_20	WSPD: None	WDIR: None
+ T303	TEMP: T_25	PRES: P	RH: RH_25	WSPD: wspd_55	WDIR: wdir_55
+ STHE	TEMP: T_20	PRES: P	RH: RH_20	WSPD: wspd_30	WDIR: wdir_30
+ IHE	TEMP: T_15	PRES: P	RH: RH_15	WSPD: None	WDIR: None


In [10]:
def adjust_df(df):
    return df.reindex(locs,axis=1)[dt_from:dt_to]

df_temp = adjust_df(df_temp)
df_pres = adjust_df(df_pres)
df_rh   = adjust_df(df_rh)
df_u    = adjust_df(df_u)
df_v    = adjust_df(df_v)
df_wspd = adjust_df(df_wspd)
df_wdir = adjust_df(df_wdir)

df_u_avg = adjust_df(df_u_avg)
df_v_avg = adjust_df(df_v_avg)
df_wspd_avg = adjust_df(df_wspd_avg)
df_wdir_avg = adjust_df(df_wdir_avg)

In [11]:
# Filter data that seems to be off...
if scope == 'H2':
    df_temp.loc[pd.to_datetime('2023-09-01 00:00', utc=True):,'D281'] = np.nan
    df_rh.loc[pd.to_datetime('2023-08-17 00:00', utc=True):,'Hintereis'] = np.nan

## Convert DataFrames to DataArrays

In [12]:
def create_data_array(df, name, long_name, standard_name, unit, time_dim='time'):
    # get time from dataset
    time = df.index
    time_naive = [dt.replace(tzinfo=None) for dt in time]

    # get data for variable (all cell IDs)
    data = df.to_numpy()

    # create DataArray
    xr_data = xr.DataArray(
        data,
        coords={
            time_dim: (time_dim, time_naive, {"long_name": "Timestamp in UTC"}),
            "location": ("location", dim_loc, {"long_name": "Location name from field campaign"}),
            "lat": ("location", dim_loc_lat, {"long_name": "Latitude", "units": "degrees_north"}),
            "lon": ("location", dim_loc_lon, {"long_name": "Longitude", "units": "degrees_east"}),
            "type": ("location", dim_loc_type, {"long_name": "Type of measuement"}),
            "ele": ("location", dim_loc_ele, {"long_name": "Elevation of location", "units": "meters"})
        },
        dims=[time_dim, "location"],
        name=name,
        attrs={
            "long_name": long_name,
            "standard_name": standard_name,
            "units": unit
        }
    )
    return xr_data

## Build dataset

In [13]:
# Combine DataArrays
ds_out = xr.Dataset({
    't_2m'  : create_data_array(df_temp, 't_2m', 'temperature in 2m', 't_2m', 'degC'),
    'rh_2m' : create_data_array(df_rh,   'rh_2m', 'relative humidity in 2m', 'rh_2m','percent'),
    'pres'  : create_data_array(df_pres, 'pres',  'surface pressure', 'surface_air_pressure', 'hPa'),  
    'u'     : create_data_array(df_u, 'u',  'Zonal wind in 2m or 3m (station dependent)', 'eastward_wind', 'm s-1'),
    'v'     : create_data_array(df_v, 'v',  'Meridional wind in 2m or 3m (station dependent)', 'northward_wind', 'm s-1'),
    'wspd'  : create_data_array(df_wspd, 'wspd',  'wind speed in 2m or 3m (station dependent)', '', 'm s-1'),  
    'wdir'  : create_data_array(df_wdir, 'wdir',  'wind direction in 2m or 3m (station dependent)', '', 'deg'),  
    
    'u_avg'     : create_data_array(df_u_avg, 'u_avg',  'Zonal wind in 2m or 3m (station dependent)', 'eastward_wind', 'm s-1', 'time_avg'),
    'v_avg'     : create_data_array(df_v_avg, 'v_avg',  'Meridional wind in 2m or 3m (station dependent)', 'northward_wind', 'm s-1', 'time_avg'),
    'wspd_avg'  : create_data_array(df_wspd_avg, 'wspd_avg',  'wind speed in 2m or 3m (station dependent)', '', 'm s-1', 'time_avg'),  
    'wdir_avg'  : create_data_array(df_wdir_avg, 'wdir_avg',  'wind direction in 2m or 3m (station dependent)', '', 'deg', 'time_avg'), 
})

# clean time series (fill gaps and remove duplicates)
ds_out = ds_out.sortby("time")
ds_out = ds_out.sel(time=~ds_out.indexes["time"].duplicated())
time_full = pd.date_range(ds_out.time.min().values, ds_out.time.max().values, freq="1h")
ds_out = ds_out.reindex(time=time_full)

ds_out = ds_out.sortby("time_avg")
ds_out = ds_out.sel(time_avg=~ds_out.indexes["time_avg"].duplicated())
time_full = pd.date_range(ds_out.time_avg.min().values, ds_out.time_avg.max().values, freq="30min")
ds_out = ds_out.reindex(time_avg=time_full)

# Add global attributes
ds_out.attrs = {
    "title": f"Observational data from {scope_name} measurement locations",
    "institution": "Humboldt-Universität zu Berlin",
    "source": f"Aggregation of various datasets from {scope_name}",
    "history": f"Created {datetime.now().strftime('%Y-%m-%d')}",
    "contact": "alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-9465-6761)",
    "campaign": scope_name,
    "StartTime": format_datetime64(ds_out.time.values[0]),
    "StopTime": format_datetime64(ds_out.time.values[-1]),
    "comment": f"Hourly data uses values at full hour (not averaged over full hour); 30min-averages cover the 30min prior to timestamp (origin='end_day')"
}

# Write
output_file = f'{scope}_Observations_stations.nc'
ds_out.to_netcdf(output_file)
print(f'Output file saved as {output_file}.')

Output file saved as H3_Observations_stations.nc.


In [14]:
ds_out

<xarray.Dataset> Size: 536kB
Dimensions:   (time: 720, location: 6, time_avg: 1441)
Coordinates:
  * time      (time) datetime64[ns] 6kB 2025-08-05 ... 2025-09-03T23:00:00
  * location  (location) object 48B 'A244' 'A245' 'T272' 'T303' 'STHE' 'IHE'
  * time_avg  (time_avg) datetime64[ns] 12kB 2025-08-05 ... 2025-09-04
    lat       (location) float64 48B 46.82 46.82 46.8 46.79 46.8 46.8
    lon       (location) float64 48B 10.81 10.79 10.77 10.75 10.76 10.78
    type      (location) object 48B 'AWS' 'AWS' 'AWS' 'AWS' 'AWS' 'AWS'
    ele       (location) float64 48B 2.438e+03 2.451e+03 ... 3.031e+03 3.264e+03
Data variables:
    t_2m      (time, location) float64 35kB nan nan nan 3.33 ... 5.356 nan nan
    rh_2m     (time, location) float64 35kB nan nan nan 94.0 ... 61.81 nan nan
    pres      (time, location) float64 35kB nan nan nan 712.1 ... 710.0 nan nan
    u         (time, location) float64 35kB nan nan nan -9.523 ... 4.729 nan nan
    v         (time, location) float64 35kB nan nan nan ... -5.197 nan nan
    wspd      (time, location) float64 35kB nan nan nan 9.59 ... 7.026 nan nan
    wdir      (time, location) float64 35kB nan nan nan 83.2 ... 317.7 nan nan
    u_avg     (time_avg, location) float64 69kB nan nan nan ... 3.937 nan nan
    v_avg     (time_avg, location) float64 69kB nan nan nan ... -6.549 nan nan
    wspd_avg  (time_avg, location) float64 69kB nan nan nan ... 7.641 nan nan
    wdir_avg  (time_avg, location) float64 69kB nan nan nan ... 329.0 nan nan
Attributes:
    title:        Observational data from HEFEX III measurement locations
    institution:  Humboldt-Universität zu Berlin
    source:       Aggregation of various datasets from HEFEX III
    history:      Created 2026-04-10
    contact:      alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-9465-...
    campaign:     HEFEX III
    StartTime:    2025-08-05 00:00:00
    StopTime:     2025-09-03 23:00:00
    comment:      Hourly data uses values at full hour (not averaged over ful...